![Logotipo de LVMM](https://lvmm.mx/wp-content/uploads/2019/05/LVMM_logo_500-90x90.png)

# Manual de uso del clúster LVMM

> Guía práctica para usuarios del Departamento de Modelación de Nanomateriales del CNyN-UNAM. Los nombres de host, las particiones, los módulos y los recursos disponibles pueden cambiar; compruébelos en el sistema antes de ejecutar sus trabajos.

## Resumen rápido
1. Conéctese mediante SSH.
2. Transfiera archivos con `sftp` o `scp`.
3. Consulte y cargue los módulos de software necesarios.
4. Use SLURM para ejecutar trabajos; para cálculos largos, prefiera `sbatch`.
5. Revise el estado de sus trabajos con `squeue`.

## Índice
- [1. Acceso al clúster](#ssh)
- [2. Transferencia de archivos](#sftp)
- [3. Módulos de software](#module)
- [4. Ejecución de trabajos con SLURM](#slurm)
  - [4.1 Consultar recursos con `sinfo`](#sinfo)
  - [4.2 Ejecutar con `srun`](#srun)
  - [4.3 Enviar trabajos con `sbatch`](#sbatch)
  - [4.4 Consultar trabajos con `squeue`](#squeue)
  - [4.5 Cancelar trabajos con `scancel`](#scancel)
  - [4.6 Sesiones interactivas con `salloc`](#salloc)
- [5. Referencias](#references)

<a name="ssh"></a>
## 1. ¿Cómo acceder al clúster LVMM?

La conexión al clúster computacional del [Departamento de Modelación de Nanomateriales](https://www.cnyn.unam.mx/?page_id=659) del [CNyN-UNAM](https://www.cnyn.unam.mx) se realiza mediante SSH. Dentro de la red del CNyN, ejecute:

```bash
ssh usuario@192.168.100.237
```

Reemplace `usuario` por el nombre de su cuenta. Al conectarse, el servidor solicitará su contraseña. Si la autenticación es correcta, verá un símbolo del sistema similar a este:

```text
usuario@lvmm:~$
```

> **Nota:** la dirección IP y el nombre del servidor son los indicados en este ejemplo. Confirme los datos de acceso vigentes con la administración del clúster.

<a name="sftp"></a>
## 2. ¿Cómo transferir información al clúster LVMM?

Para transferir archivos entre su equipo y el clúster, puede utilizar `sftp` o `scp`, ambos basados en SSH.

### Transferencia interactiva con `sftp`

```bash
sftp usuario@192.168.100.237
```

Una vez conectado, use los comandos de `sftp` para listar, subir o descargar archivos.

### Copia con `scp`

La sintaxis general es `scp [opciones] origen destino`. Una ruta remota tiene la forma `usuario@servidor:ruta`. Para copiar un directorio remoto al directorio local actual, use `-r`:

```bash
scp -r usuario@192.168.100.237:/tmpu/grupo/usuario/datos .
```

Para copiar un archivo local al clúster:

```bash
scp archivo.txt usuario@192.168.100.237:/tmpu/grupo/usuario/
```

> **Consejo:** compruebe la ruta de destino y los permisos de escritura antes de transferir archivos.

<a name="module"></a>
## 3. ¿Cómo utilizar los programas instalados?

El clúster utiliza [Environment Modules](https://modules.sourceforge.net/) para administrar las aplicaciones y sus dependencias.

### Consultar los módulos disponibles

```bash
module available
# Forma abreviada:
ml av
```

La lista que devuelve el comando puede cambiar con el tiempo. Consulte la salida actual en el clúster, en lugar de depender de una lista guardada en esta guía.

### Cargar un módulo

Por ejemplo, para cargar la versión de VASP para GPU que aparece en el listado de módulos:

```bash
ml load vasp/6.4.3-nvhpc-gpu
```

El sistema puede mostrar mensajes indicando que también cargó dependencias.

> **Importante:** no ejecute cálculos intensivos directamente en el nodo de acceso (`login node`). Use los recursos de cómputo mediante SLURM. Para un trabajo real, solicite la partición y los recursos compatibles con el programa y el módulo. La forma de solicitar GPU depende de la configuración local; confirme las opciones con la documentación del clúster o su administración.

<a name="slurm"></a>
## 4. ¿Cómo ejecutar una aplicación con SLURM?

El clúster HPC LVMM utiliza [SLURM](https://slurm.schedmd.com), un sistema de administración y planificación de trabajos. El flujo habitual es consultar los recursos, enviar el trabajo y revisar su estado.

> Para trabajos de larga duración o producción, se recomienda `sbatch`. `srun` es útil para ejecuciones interactivas o dentro de una asignación de recursos.

<a name="sinfo"></a>
### 4.1 Consultar recursos con `sinfo`

`sinfo` muestra el estado de las particiones y los nodos disponibles:

```bash
sinfo
```

Para consultar la información por nodo:

```bash
sinfo --Node
```

Los estados y las particiones mostrados dependen de la configuración y disponibilidad actuales. Consulte `sinfo` antes de enviar un trabajo y confirme con la administración qué partición corresponde a su necesidad.

<a name="srun"></a>
### 4.2 Ejecutar con `srun`

`srun` permite iniciar una tarea con recursos de SLURM. Este ejemplo reproduce el comando usado en la configuración documentada del clúster; ajuste partición y número de tareas a los requisitos de su aplicación:

```bash
srun --partition=q_mem --ntasks=16 vasp_ncl
```

La ejecución puede esperar hasta que los recursos solicitados estén disponibles y mantiene la terminal ocupada mientras corre. Para enviar un trabajo y dejarlo en la cola, use `sbatch`.

<a name="sbatch"></a>
### 4.3 Enviar trabajos con `sbatch`

Prepare un script de trabajo, por ejemplo `vasp.job`. Los valores de recursos y la partición son ilustrativos: verifique que sean adecuados para su caso y para la política vigente del clúster.

```bash
#!/usr/bin/env bash
#SBATCH --job-name=VASP_test
#SBATCH --nodes=1
#SBATCH --ntasks=16
#SBATCH --cpus-per-task=1
#SBATCH --output=vasp_test-%j.out
#SBATCH --error=vasp_test-%j.err
#SBATCH --partition=q_cpu

. /etc/profile.d/modules.sh
ml vasp/6.4.3-intel

srun vasp_ncl
```

Envíe el script con:

```bash
sbatch vasp.job
```

SLURM responderá con un identificador de trabajo (`JOBID`). Los archivos de salida y error se crearán con el identificador incluido en sus nombres (`%j`).

> **Antes de enviar:** confirme que la partición, el módulo y el número de tareas coinciden con su aplicación. Para GPU, solicite los recursos con las opciones que admita la configuración local; no asuma una opción de GPU sin verificarla.

<a name="squeue"></a>
### 4.4 Consultar trabajos con `squeue`

Para ver los trabajos en cola, ejecute:

```bash
squeue
```

Para mostrar solo los trabajos de su cuenta:

```bash
squeue --user $USER
```

La columna `ST` indica el estado del trabajo; por ejemplo, `R` significa que está corriendo y `PD` que está pendiente. Si un trabajo permanece pendiente, consulte la razón que muestra SLURM en la columna correspondiente.

<a name="scancel"></a>
### 4.5 Cancelar trabajos con `scancel`

Para cancelar un trabajo, use su identificador:

```bash
scancel JOBID
```

También puede cancelar por nombre con `scancel --name NOMBRE`, teniendo cuidado de no detener otros trabajos con el mismo nombre. Verifique que el identificador o el nombre corresponda al trabajo que desea cancelar.

<a name="salloc"></a>
### 4.6 Sesiones interactivas con `salloc`

`salloc` solicita recursos para una sesión interactiva. Ejemplo:

```bash
salloc -p q_cpu -N 1 -n 16 -J VASP_test -t 60
```

Una vez concedida la asignación, ejecute el programa mediante `srun` dentro de la sesión, por ejemplo:

```bash
ml vasp/6.4.3-intel
srun vasp_ncl
```

Al terminar, cierre la sesión con `exit` o `Ctrl + D`. `salloc` es apropiado para pruebas interactivas; para ejecuciones largas y reproducibles, prefiera `sbatch`.

<a name="references"></a>
---
## 5. Referencias

- [Sitio web de SLURM](https://slurm.schedmd.com/)
- [Documentación de SLURM](https://slurm.schedmd.com/documentation.html)
- [Comunidad de usuarios de SLURM](https://groups.google.com/g/slurm-users)
- [Environment Modules](https://modules.sourceforge.net/)

Aldo Rodríguez Guerrero, 2024–2026